# FASE 2. LIMPIEZA DE LOS DATOS - CALIDAD DEL AIRE (EEA)

## 1. Carga de datos originales

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("TFM_Limpieza_Aire_EEA") \
    .getOrCreate()

# Ruta de los archivos originales
ruta_eea = "gs://tfm_unir1/aire/eea/*.parquet"

# Cargar los datos
df_raw = spark.read.parquet(ruta_eea)

print("Registros originales:", df_raw.count())
print("Columnas:", len(df_raw.columns))

26/10/21 11:22:03 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Registros originales: 7444091
Columnas: 12


## 2. Selección de contaminantes de interés 

In [2]:
# Códigos EEA de los contaminantes seleccionados
contaminantes = {
    5: "PM10",
    7: "O3",
    8: "NO2",
    6001: "PM2.5"
}

# Crear un nuevo DataFrame con los cuatro contaminantes
df_aire = df_raw.filter(F.col("Pollutant").isin(list(contaminantes.keys())))

# Comprobar los registros seleccionados
df_aire.groupBy("Pollutant") \
    .count() \
    .orderBy("Pollutant") \
    .show()

print("Total de registros seleccionados:", df_aire.count())

+---------+-------+
|Pollutant|  count|
+---------+-------+
|        5| 605201|
|        7|1081406|
|        8|1106507|
|     6001| 591819|
+---------+-------+



Total de registros seleccionados: 3384933


## 3. Separación de los datos según su frecuencia temporal

In [3]:
# Separar registros por frecuencia temporal
df_horario = df_aire.filter(F.col("AggType") == "hour")

df_diario = df_aire.filter(F.col("AggType") == "day")

# Comprobar los resultados
print("Registros horarios:", df_horario.count())
print("Registros diarios:", df_diario.count())

print("Total:", df_horario.count() + df_diario.count())

Registros horarios: 3362087


Registros diarios: 22846


Total: 3384933


## 4. Limpieza de los datos horarios

### Tratamiento de valores no válidos

In [4]:
from pyspark.sql import functions as F

# Crear una columna con la concentración limpia
df_horario_clean = df_horario.withColumn(
    "Value_clean",
    F.when(
        ((F.col("Validity") == 1) |
         ((F.col("Pollutant") == 7) & (F.col("Validity") == 4))) &
        (F.col("Value") != -999),
        F.col("Value")
    ).otherwise(None))

# Comprobar cuántos registros quedan con valor disponible
df_horario_clean.groupBy("Pollutant").agg(
    F.count("*").alias("total_registros"),
    F.count("Value_clean").alias("valores_utilizables"),
    F.sum(
        F.when(F.col("Value_clean").isNull(), 1).otherwise(0)
    ).alias("valores_no_utilizables")
).orderBy("Pollutant").show()

+---------+---------------+-------------------+----------------------+
|Pollutant|total_registros|valores_utilizables|valores_no_utilizables|
+---------+---------------+-------------------+----------------------+
|        5|         587092|             547904|                 39188|
|        7|        1081406|            1034362|                 47044|
|        8|        1106507|            1028982|                 77525|
|     6001|         587082|             547482|                 39600|
+---------+---------------+-------------------+----------------------+



### Verificación de concentraciones tras la limpieza

In [6]:
df_horario_clean.groupBy("Pollutant").agg(
    F.count("Value_clean").alias("valores_utilizables"),
    F.min("Value_clean").alias("valor_minimo"),
    F.max("Value_clean").alias("valor_maximo"),
    F.sum(F.when(F.col("Value_clean") < 0, 1).otherwise(0)
    ).alias("valores_negativos")
).orderBy("Pollutant").show(truncate=False)

+---------+-------------------+------------+----------------------+-----------------+
|Pollutant|valores_utilizables|valor_minimo|valor_maximo          |valores_negativos|
+---------+-------------------+------------+----------------------+-----------------+
|5        |547904             |0E-18       |831.000000000000000000|0                |
|7        |1034362            |0E-18       |178.000000000000000000|0                |
|8        |1028982            |0E-18       |285.000000000000000000|0                |
|6001     |547482             |0E-18       |846.000000000000000000|0                |
+---------+-------------------+------------+----------------------+-----------------+



## 5. Limpieza de los datos diarios

### Tratamiento de los valores no válidos

In [7]:
df_diario_clean = df_diario.withColumn(
    "Value_clean",
    F.when(
        (F.col("Validity") == 1) &
        (F.col("Value").isNotNull()) &
        (F.col("Value") != -999),
        F.col("Value")
    ).otherwise(None))

df_diario_clean.groupBy("Pollutant").agg(
    F.count("*").alias("total_registros"),
    F.count("Value_clean").alias("valores_utilizables"),
    F.sum(
        F.when(F.col("Value_clean").isNull(), 1).otherwise(0)
    ).alias("valores_no_utilizables")
).orderBy("Pollutant").show()

+---------+---------------+-------------------+----------------------+
|Pollutant|total_registros|valores_utilizables|valores_no_utilizables|
+---------+---------------+-------------------+----------------------+
|        5|          18109|              16768|                  1341|
|     6001|           4737|               4036|                   701|
+---------+---------------+-------------------+----------------------+



## 6. Estandarización de variables 

In [8]:
from pyspark.sql import functions as F

# Diccionario de contaminantes
nombres = {
    5: "PM10",
    7: "O3",
    8: "NO2",
    6001: "PM2.5"
}

# Crear expresión de correspondencia
mapa = F.create_map(
    *[x for codigo, nombre in nombres.items()
      for x in (F.lit(codigo), F.lit(nombre))])

# Añadir nombres a ambos DataFrames
df_horario_clean = df_horario_clean.withColumn(
    "contaminante", mapa[F.col("Pollutant")])

df_diario_clean = df_diario_clean.withColumn(
    "contaminante", mapa[F.col("Pollutant")])

# Verificar resultado
df_horario_clean.select(
    "Pollutant", "contaminante").distinct().orderBy("Pollutant").show()

+---------+------------+
|Pollutant|contaminante|
+---------+------------+
|        5|        PM10|
|        7|          O3|
|        8|         NO2|
|     6001|       PM2.5|
+---------+------------+



### Estandarización de las variables temporales

In [9]:
from pyspark.sql import functions as F

# Añadir variables temporales a los datos horarios
df_horario_clean = (
    df_horario_clean
    .withColumn("anio", F.year("Start"))
    .withColumn("mes", F.month("Start"))
    .withColumn("dia", F.dayofmonth("Start"))
    .withColumn("hora", F.hour("Start"))
    .withColumn("dia_semana", F.dayofweek("Start"))
)

# Añadir variables temporales a los datos diarios
df_diario_clean = (
    df_diario_clean
    .withColumn("anio", F.year("Start"))
    .withColumn("mes", F.month("Start"))
    .withColumn("dia", F.dayofmonth("Start"))
)

# Comprobar resultado
df_horario_clean.select(
    "Start", "anio", "mes", "dia", "hora", "dia_semana"
).show(5, truncate=False)

+-------------------+----+---+---+----+----------+
|Start              |anio|mes|dia|hora|dia_semana|
+-------------------+----+---+---+----+----------+
|2013-01-01 01:00:00|2013|1  |1  |1   |3         |
|2013-01-01 02:00:00|2013|1  |1  |2   |3         |
|2013-01-01 03:00:00|2013|1  |1  |3   |3         |
|2013-01-01 04:00:00|2013|1  |1  |4   |3         |
|2013-01-01 05:00:00|2013|1  |1  |5   |3         |
+-------------------+----+---+---+----+----------+
only showing top 5 rows



### Verificación de la zona horaria

In [12]:
# Consultar la zona horaria utilizada por Spark
zona_horaria = spark.conf.get("spark.sql.session.timeZone")

print("Zona horaria de Spark:", zona_horaria)

Zona horaria de Spark: Etc/UTC


### Normalización de nombres de columnas

In [14]:
# Diccionario de nombres originales y estandarizados
columnas = {
    "Samplingpoint": "id_punto_medicion",
    "Pollutant": "codigo_contaminante",
    "Start": "fecha_inicio",
    "End": "fecha_fin",
    "Value": "valor_original",
    "Value_clean": "concentracion",
    "Unit": "unidad",
    "AggType": "frecuencia",
    "Validity": "validez",
    "Verification": "verificacion"}

# Función para renombrar columnas
def renombrar_columnas(df):
    for original, nuevo in columnas.items():
        df = df.withColumnRenamed(original, nuevo)
    return df

# Aplicar a ambos conjuntos
df_horario_clean = renombrar_columnas(df_horario_clean)
df_diario_clean = renombrar_columnas(df_diario_clean)

# Comprobar las columnas resultantes
print("Columnas del dataset horario:")
print(df_horario_clean.columns)

print("\nColumnas del dataset diario:")
print(df_diario_clean.columns)

Columnas del dataset horario:
['id_punto_medicion', 'codigo_contaminante', 'fecha_inicio', 'fecha_fin', 'valor_original', 'unidad', 'frecuencia', 'validez', 'verificacion', 'ResultTime', 'DataCapture', 'FkObservationLog', 'concentracion', 'contaminante', 'anio', 'mes', 'dia', 'hora', 'dia_semana']

Columnas del dataset diario:
['id_punto_medicion', 'codigo_contaminante', 'fecha_inicio', 'fecha_fin', 'valor_original', 'unidad', 'frecuencia', 'validez', 'verificacion', 'ResultTime', 'DataCapture', 'FkObservationLog', 'concentracion', 'contaminante', 'anio', 'mes', 'dia']


## 7. Validación final de los datos preparados

In [15]:
# Comparar los registros antes y después de la limpieza
resultados = [
    ("Horarios", df_horario.count(), df_horario_clean.count()),
    ("Diarios", df_diario.count(), df_diario_clean.count())
]

for tipo, originales, limpios in resultados:
    print(f"\nDatos {tipo}")
    print("Registros originales:", originales)
    print("Registros preparados:", limpios)
    print("Diferencia:", originales - limpios)

    if originales == limpios:
        print("OK: No se han perdido registros")
    else:
        print("REVISAR: Existen diferencias")


Datos Horarios
Registros originales: 3362087
Registros preparados: 3362087
Diferencia: 0
OK: No se han perdido registros

Datos Diarios
Registros originales: 22846
Registros preparados: 22846
Diferencia: 0
OK: No se han perdido registros


## 8. Exportación de los datasets limpios

In [16]:
# Rutas de salida
ruta_horario = "gs://tfm_unir1/processed/aire/eea/horario/"
ruta_diario = "gs://tfm_unir1/processed/aire/eea/diario/"

# Guardar datos horarios
df_horario_clean.write \
    .mode("errorifexists") \
    .parquet(ruta_horario)

# Guardar datos diarios
df_diario_clean.write \
    .mode("errorifexists") \
    .parquet(ruta_diario)

print("Datos limpios guardados correctamente")

Datos limpios guardados correctamente
